In [19]:
! pip install implicit -q

In [20]:
import pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
import polars as pl
import numpy as np
from collections import defaultdict
from typing import Dict, List
import sys

CANDGEN_MODELS_DIR = "/kaggle/input/datasets/fedorfesenko/candgen-models"
sys.path.append("/kaggle/input/models/fedorfesenko/metrcs/other/default/1")
from metrics import evaluate_candidates

In [21]:
class BaseTwoTowerModel(nn.Module):
    def __init__(self, n_users, n_items, emb_dim):
        super().__init__()
        self.user_embeds = nn.Embedding(n_users, emb_dim)
        self.item_embeds = nn.Embedding(n_items, emb_dim)
    
    def forward(self, inputs):
        if self.training:
            return self.compute_loss(inputs)
        uid = inputs["uid"]
        user_emb = self.user_embeds(uid)
        item_embs = self.item_embeds.weight
        return user_emb @ item_embs.T


class InBatchLogQMF(BaseTwoTowerModel):
    def __init__(self, n_users, n_items, emb_dim, item_counts=None):
        super().__init__(n_users, n_items, emb_dim)
        if item_counts is not None:
            item_probs = item_counts / item_counts.sum()
            log_q = torch.log(item_probs + 1e-10)
            self.register_buffer("log_q", log_q)
    
    def compute_loss(self, inputs):
        user_emb = self.user_embeds(inputs["uid"])
        labels = inputs["label"]
        item_embs = self.item_embeds(labels)
        scores = user_emb @ item_embs.T
        log_q_batch = self.log_q[labels]
        scores = scores - log_q_batch.unsqueeze(0)
        B = user_emb.shape[0]
        targets = torch.arange(B, device=user_emb.device)
        return F.cross_entropy(scores, targets)


class MeanUserEncoder(nn.Module):
    def __init__(self, n_items, emb_dim):
        super().__init__()
        self.item_embeddings = nn.Embedding(n_items, emb_dim)
    
    def forward(self, history):
        item_idx = history["item_id"]
        length = history["length"]
        B = length.shape[0]
        item_embs = self.item_embeddings(item_idx)
        user_indices = torch.repeat_interleave(
            torch.arange(B, device=item_idx.device), length
        )
        emb_dim = self.item_embeddings.embedding_dim
        user_emb = torch.zeros(B, emb_dim, device=item_idx.device)
        user_emb.scatter_add_(
            0,
            user_indices.unsqueeze(1).expand(-1, emb_dim),
            item_embs
        )
        user_emb = user_emb / length.unsqueeze(1)
        return user_emb


class BaseMeanTwoTowerModel(nn.Module):
    def __init__(self, n_items, emb_dim):
        super().__init__()
        self.encoder = MeanUserEncoder(n_items, emb_dim)
    
    def forward(self, inputs):
        if self.training:
            return self.compute_loss(inputs)
        user_emb = self.encoder(inputs["history"])
        item_embs = self.encoder.item_embeddings.weight
        return user_emb @ item_embs.T


class InBatchLogQMeanTwoTowerModel(BaseMeanTwoTowerModel):
    def __init__(self, n_items, emb_dim, item_counts=None):
        super().__init__(n_items, emb_dim)
        if item_counts is not None:
            item_probs = item_counts / item_counts.sum()
            log_q = torch.log(item_probs + 1e-10)
            self.register_buffer("log_q", log_q)
    
    def compute_loss(self, inputs):
        user_emb = self.encoder(inputs["history"])
        labels = inputs["label"]
        item_embs = self.encoder.item_embeddings(labels)
        scores = user_emb @ item_embs.T
        log_q_batch = self.log_q[labels]
        scores = scores - log_q_batch.unsqueeze(0)
        B = user_emb.shape[0]
        targets = torch.arange(B, device=user_emb.device)
        return F.cross_entropy(scores, targets)

In [22]:
# i2i
with open(f"{CANDGEN_MODELS_DIR}/i2i_artifacts.pkl", "rb") as f:
    i2i_data = pickle.load(f)

similar_items = i2i_data["similar_items"]
user_to_idx = i2i_data["user_to_idx"]
item_to_idx = i2i_data["item_to_idx"]
idx_to_item = i2i_data["idx_to_item"]
pop_items = i2i_data["pop_items"]

print(f"i2i: {len(similar_items)} items, {len(user_to_idx)} users, {len(item_to_idx)} items")

i2i: 13106 items, 2211 users, 13106 items


In [23]:
# ALS
with open(f"{CANDGEN_MODELS_DIR}/als_artifacts.pkl", "rb") as f:
    als_data = pickle.load(f)

als_model = als_data["model"]
R = als_data["R"]

print(f"ALS: {type(als_model).__name__}")
print(f"  user_factors: {als_model.user_factors.shape if als_model.user_factors is not None else 'None'}")
print(f"  R: {R.shape}, nnz={R.nnz:,}")

ALS: AlternatingLeastSquares
  user_factors: (2211, 256)
  R: (2211, 13106), nnz=132,226


In [24]:
# two tower model v5
checkpoint_v5 = torch.load(f"{CANDGEN_MODELS_DIR}/two_tower_v5_logq.pth", map_location="cpu")
print(f"\nv5 keys: {list(checkpoint_v5.keys())}")


v5 keys: ['model_state_dict', 'model_class', 'config', 'user_to_idx', 'item_to_idx', 'idx_to_item', 'n_users', 'n_items', 'pop_items']


In [25]:
# v6
checkpoint_v6 = torch.load(f"{CANDGEN_MODELS_DIR}/mf_v6_inbatch_logq.pth", map_location="cpu")
print(f"v6 keys: {list(checkpoint_v6.keys())}")


v6 keys: ['model_state_dict', 'model_class', 'config', 'user_to_idx', 'item_to_idx', 'idx_to_item', 'n_users', 'n_items', 'pop_items']


In [26]:
# Маппинги из checkpoint
user_to_idx_v6 = checkpoint_v6["user_to_idx"]
item_to_idx_v6 = checkpoint_v6["item_to_idx"]
idx_to_item_v6 = checkpoint_v6["idx_to_item"]

print(f"\nuser_to_idx_v6: {len(user_to_idx_v6)}")
print(f"item_to_idx_v6: {len(item_to_idx_v6)}")


user_to_idx_v6: 2245
item_to_idx_v6: 13179


In [27]:
# two tower model v6
model_v6 = InBatchLogQMF(
    n_users=checkpoint_v6["config"]["n_users"],
    n_items=checkpoint_v6["config"]["n_items"],
    emb_dim=checkpoint_v6["config"]["emb_dim"],
    item_counts=torch.ones(checkpoint_v6["config"]["n_items"]),
)
model_v6.load_state_dict(checkpoint_v6["model_state_dict"])
model_v6.eval()
print("v6 загружен")

v6 загружен


In [28]:
# two tower model v5
model_v5 = InBatchLogQMeanTwoTowerModel(
    n_items=checkpoint_v5["config"]["n_items"],
    emb_dim=checkpoint_v5["config"]["emb_dim"],
    item_counts=torch.ones(checkpoint_v5["config"]["n_items"]),
)
model_v5.load_state_dict(checkpoint_v5["model_state_dict"])
model_v5.eval()
print("v5 загружен")

v5 загружен


In [29]:
df_train = pl.read_parquet("/kaggle/input/datasets/fedorfesenko/train-interactions/df_interactions_train.parquet")
df_val = pl.read_parquet("/kaggle/input/datasets/fedorfesenko/train-interactions/df_interactions_val.parquet")

train_histories = {
    row["user_id"]: row["items"]
    for row in (
        df_train
        .filter(pl.col("action_type") == "AT_CartUpdate")
        .group_by("user_id")
        .agg(pl.col("product_id").sort_by("timestamp").alias("items"))
    ).iter_rows(named=True)
}

true_items_val = {
    row["user_id"]: set(row["items"])
    for row in (
        df_val
        .filter(pl.col("action_type") == "AT_CartUpdate")
        .group_by("user_id")
        .agg(pl.col("product_id").alias("items"))
    ).iter_rows(named=True)
}

print(f"train_histories: {len(train_histories)}")
print(f"true_items_val: {len(true_items_val)}")

train_histories: 2211
true_items_val: 432


In [30]:
def predict_i2i(user_history, user_ids, similar_items, k=500):
    predictions = {}
    for user_id in user_ids:
        if user_id not in user_history:
            continue
        scores = defaultdict(float)
        for item_id in user_history[user_id]:
            if item_id not in item_to_idx:
                continue
            item_idx = item_to_idx[item_id]
            for neighbor_idx, sim_score in similar_items.get(item_idx, []):
                neighbor_item = idx_to_item[neighbor_idx]
                scores[neighbor_item] += sim_score
        sorted_items = sorted(scores.items(), key=lambda x: -x[1])
        predictions[user_id] = [item for item, _ in sorted_items[:k]]
    return predictions


def predict_als(user_ids, als_model, R, k=500):
    predictions = {}
    for user_id in user_ids:
        if user_id not in user_to_idx:
            predictions[user_id] = []
            continue
        user_idx = user_to_idx[user_id]
        try:
            ids, _ = als_model.recommend(
                userid=user_idx,
                user_items=R[user_idx],
                N=k,
                filter_already_liked_items=False,
            )
            predictions[user_id] = [idx_to_item[i] for i in ids]
        except Exception as e:
            predictions[user_id] = []
    return predictions


def predict_v6(user_ids, model_v6, k=500, batch_size=512):
    model_v6.eval()
    predictions = {}
    user_ids_list = list(user_ids)
    
    with torch.no_grad():
        for i in range(0, len(user_ids_list), batch_size):
            batch_uids = user_ids_list[i:i+batch_size]
            
            uids_idx = []
            valid_uids = []
            for uid in batch_uids:
                if uid in user_to_idx_v6:
                    uids_idx.append(user_to_idx_v6[uid])
                    valid_uids.append(uid)
            
            if not uids_idx:
                continue
            
            uids_tensor = torch.tensor(uids_idx, dtype=torch.long)
            user_emb = model_v6.user_embeds(uids_tensor)
            item_embs = model_v6.item_embeds.weight
            scores = user_emb @ item_embs.T
            
            _, top_k = scores.topk(k, dim=1)
            top_k = top_k.cpu().numpy()
            
            for j, uid in enumerate(valid_uids):
                predictions[uid] = [idx_to_item_v6[int(idx)] for idx in top_k[j]]
    
    return predictions


def predict_v5(user_ids, model_v5, user_history, k=500):
    model_v5.eval()
    predictions = {}
    item_embs = model_v5.encoder.item_embeddings.weight
    
    with torch.no_grad():
        for uid in user_ids:
            if uid not in user_history:
                continue
            
            history_items = user_history[uid]
            item_idxs = [item_to_idx_v6[i] for i in history_items if i in item_to_idx_v6]
            
            if not item_idxs:
                continue
            
            item_idxs_tensor = torch.tensor(item_idxs, dtype=torch.long)
            length_tensor = torch.tensor([len(item_idxs)], dtype=torch.long)
            
            user_emb = model_v5.encoder({
                "item_id": item_idxs_tensor,
                "length": length_tensor,
            })
            
            scores = user_emb @ item_embs.T
            _, top_k = scores.topk(k, dim=1)
            top_k = top_k.cpu().numpy()[0]
            
            predictions[uid] = [idx_to_item_v6[int(idx)] for idx in top_k]
    
    return predictions

In [31]:
val_users = list(true_items_val.keys())
print(f"Val users: {len(val_users)}")

print("\nPredicting i2i...")
pred_i2i = predict_i2i(train_histories, val_users, similar_items, k=500)
print(f"  i2i: {len(pred_i2i)} users")

print("Predicting ALS...")
pred_als = predict_als(val_users, als_model, R, k=500)
empty_als = sum(1 for p in pred_als.values() if len(p) == 0)
print(f"  ALS: {len(pred_als)} users, empty: {empty_als}")

print("Predicting v6...")
pred_v6 = predict_v6(val_users, model_v6, k=500)
print(f"  v6: {len(pred_v6)} users")

print("Predicting v5...")
pred_v5 = predict_v5(val_users, model_v5, train_histories, k=500)
print(f"  v5: {len(pred_v5)} users")

Val users: 432

Predicting i2i...
  i2i: 398 users
Predicting ALS...
  ALS: 432 users, empty: 34
Predicting v6...
  v6: 432 users
Predicting v5...
  v5: 398 users


In [37]:
def union_candidates_interleaved(predictions_list, k=500):
    all_users = set()
    for preds in predictions_list:
        all_users.update(preds.keys())
    
    union = {}
    for user_id in all_users:
        per_model = [preds.get(user_id, []) for preds in predictions_list]
        combined = []
        max_len = max(len(p) for p in per_model) if per_model else 0
        
        for i in range(max_len):
            for p in per_model:
                if i < len(p):
                    combined.append(p[i])
                if len(combined) >= k * 2:
                    break
            if len(combined) >= k * 2:
                break
        
        union[user_id] = list(dict.fromkeys(combined))[:k]
    
    return union

pred_union = union_candidates_interleaved([pred_i2i, pred_als, pred_v6, pred_v5], k=500)

# Fallback
for uid in val_users:
    if uid not in pred_union or len(pred_union[uid]) == 0:
        pred_union[uid] = pop_items

print(f"Union: {len(pred_union)} users")

result_union = evaluate_candidates(
    pred_union,
    true_items_val,
    k=500,
    total_items=len(idx_to_item_v6),
)

for name, value in result_union.items():
    if isinstance(value, float):
        print(f"  {name}: {value:.4f}")
    else:
        print(f"  {name}: {value}")

Union: 432 users
  recall@500: 0.5627
  hit_rate@500: 0.8542
  mrr@500: 0.1751
  coverage: 0.9483
  n_requests: 432
  avg_candidates: 499.6227


In [44]:
class CandidateGenerator:
    def __init__(
        self,
        # i2i
        similar_items,
        user_to_idx,
        item_to_idx,
        idx_to_item,
        pop_items,
        # ALS
        als_model,
        R,
        # v5
        model_v5,
        # v6
        model_v6,
        user_to_idx_v6,
        item_to_idx_v6,
        idx_to_item_v6,
        # device
        device="cpu",
    ):
        # i2i
        self.similar_items = similar_items
        self.user_to_idx = user_to_idx
        self.item_to_idx = item_to_idx
        self.idx_to_item = idx_to_item
        self.pop_items = pop_items
        
        # ALS
        self.als_model = als_model
        self.R = R
        
        # v5
        self.model_v5 = model_v5
        
        # v6
        self.model_v6 = model_v6
        self.user_to_idx_v6 = user_to_idx_v6
        self.item_to_idx_v6 = item_to_idx_v6
        self.idx_to_item_v6 = idx_to_item_v6
        
        # device
        self.device = device
        self.model_v5 = model_v5.to(device)
        self.model_v6 = model_v6.to(device)
    
    # === I2I ===
    def predict_i2i(self, user_history, user_ids, k=500):
        predictions = {}
        for user_id in user_ids:
            if user_id not in user_history:
                continue
            scores = defaultdict(float)
            for item_id in user_history[user_id]:
                if item_id not in self.item_to_idx:
                    continue
                item_idx = self.item_to_idx[item_id]
                for neighbor_idx, sim_score in self.similar_items.get(item_idx, []):
                    neighbor_item = self.idx_to_item[neighbor_idx]
                    scores[neighbor_item] += sim_score
            sorted_items = sorted(scores.items(), key=lambda x: -x[1])
            predictions[user_id] = [item for item, _ in sorted_items[:k]]
        return predictions
    
    # === ALS ===
    def predict_als(self, user_ids, k=500):
        predictions = {}
        for user_id in user_ids:
            if user_id not in self.user_to_idx:
                predictions[user_id] = []
                continue
            user_idx = self.user_to_idx[user_id]
            try:
                ids, _ = self.als_model.recommend(
                    userid=user_idx,
                    user_items=self.R[user_idx],
                    N=k,
                    filter_already_liked_items=False,
                )
                predictions[user_id] = [self.idx_to_item[i] for i in ids]
            except Exception:
                predictions[user_id] = []
        return predictions
    
    # === v6: MF ===
    def predict_v6(self, user_ids, k=500, batch_size=512):
        self.model_v6.eval()
        predictions = {}
        user_ids_list = list(user_ids)
        
        with torch.no_grad():
            for i in range(0, len(user_ids_list), batch_size):
                batch_uids = user_ids_list[i:i+batch_size]
                
                uids_idx = []
                valid_uids = []
                for uid in batch_uids:
                    if uid in self.user_to_idx_v6:
                        uids_idx.append(self.user_to_idx_v6[uid])
                        valid_uids.append(uid)
                
                if not uids_idx:
                    continue
                
                uids_tensor = torch.tensor(uids_idx, dtype=torch.long, device=self.device)
                user_emb = self.model_v6.user_embeds(uids_tensor)
                item_embs = self.model_v6.item_embeds.weight
                scores = user_emb @ item_embs.T
                
                _, top_k = scores.topk(k, dim=1)
                top_k = top_k.cpu().numpy()
                
                for j, uid in enumerate(valid_uids):
                    predictions[uid] = [
                        self.idx_to_item_v6[int(idx)] for idx in top_k[j]
                    ]
        
        return predictions
    
    # === v5: Two-Tower ===
    def predict_v5(self, user_ids, user_history, k=500):
        self.model_v5.eval()
        predictions = {}
        item_embs = self.model_v5.encoder.item_embeddings.weight
        
        with torch.no_grad():
            for uid in user_ids:
                if uid not in user_history:
                    continue
                
                history_items = user_history[uid]
                item_idxs = [
                    self.item_to_idx_v6[i] for i in history_items
                    if i in self.item_to_idx_v6
                ]
                
                if not item_idxs:
                    continue
                
                item_idxs_tensor = torch.tensor(
                    item_idxs, dtype=torch.long, device=self.device
                )
                length_tensor = torch.tensor(
                    [len(item_idxs)], dtype=torch.long, device=self.device
                )
                
                user_emb = self.model_v5.encoder({
                    "item_id": item_idxs_tensor,
                    "length": length_tensor,
                })
                
                scores = user_emb @ item_embs.T
                _, top_k = scores.topk(k, dim=1)
                top_k = top_k.cpu().numpy()[0]
                
                predictions[uid] = [
                    self.idx_to_item_v6[int(idx)] for idx in top_k
                ]
        
        return predictions
    
    def union_interleaved(self, predictions_list, k=500):
        all_users = set()
        for preds in predictions_list:
            all_users.update(preds.keys())
        
        union = {}
        for user_id in all_users:
            per_model = [preds.get(user_id, []) for preds in predictions_list]
            combined = []
            max_len = max(len(p) for p in per_model) if per_model else 0
            
            for i in range(max_len):
                for p in per_model:
                    if i < len(p):
                        combined.append(p[i])
                    if len(combined) >= k * 2:
                        break
                if len(combined) >= k * 2:
                    break
            
            union[user_id] = list(dict.fromkeys(combined))[:k]
        
        return union
    
    # === Полный пайплайн ===
    def predict(self, user_ids, user_history, k=500, rrf_k=60):
        pred_i2i = self.predict_i2i(user_history, user_ids, k=k)
        pred_als = self.predict_als(user_ids, k=k)
        pred_v6 = self.predict_v6(user_ids, k=k)
        pred_v5 = self.predict_v5(user_ids, user_history, k=k)
        
        union = self.union_interleaved(
            [pred_i2i, pred_als, pred_v6, pred_v5],
            k=k
        )
        
        # Fallback
        for uid in user_ids:
            if uid not in union or len(union[uid]) == 0:
                union[uid] = self.pop_items
        
        return union

In [45]:
candgen = CandidateGenerator(
    # i2i
    similar_items=similar_items,
    user_to_idx=user_to_idx,
    item_to_idx=item_to_idx,
    idx_to_item=idx_to_item,
    pop_items=pop_items,
    # ALS
    als_model=als_model,
    R=R,
    # v5
    model_v5=model_v5,
    # v6
    model_v6=model_v6,
    user_to_idx_v6=user_to_idx_v6,
    item_to_idx_v6=item_to_idx_v6,
    idx_to_item_v6=idx_to_item_v6,
    # device
    device="cpu",
)


val_users = list(true_items_val.keys())
pred_union = candgen.predict(val_users, train_histories, k=500)

result = evaluate_candidates(
    pred_union,
    true_items_val,
    k=500,
    total_items=len(idx_to_item_v6),
)


In [46]:
result

{'recall@500': 0.5626818648681037,
 'hit_rate@500': 0.8541666666666666,
 'mrr@500': 0.1751069554663251,
 'coverage': 0.9482510053873586,
 'n_requests': 432,
 'avg_candidates': 499.62268518518516}